# Image Classification 

This notebook explores Image Classification as a Linear Classifier, Neural Network, and then a Convoluted Neural Network (CNN) 

## Dataset

The dataset is a bunch of pictures of flowers, labelled by type

### Example 
| File | 0001.png                                          | 0003.png                                          | 0006.png                                          | 0206.png                                          | 0207.png                                          | 0209.png                                          |
| ----- | ------------------------------------------------- | ------------------------------------------------- | ------------------------------------------------- | ------------------------------------------------- | ------------------------------------------------- | ------------------------------------------------- |
| Label | 0                                                 | 2                                                 | 1                                                 | 6                                                 | 0                                                 | 6                                                 |
| Image | ![flower_images/0001.png](flower_images/0001.png) | ![flower_images/0001.png](flower_images/0003.png) | ![flower_images/0001.png](flower_images/0006.png) | ![flower_images/0001.png](flower_images/0206.png) | ![flower_images/0001.png](flower_images/0207.png) | ![flower_images/0001.png](flower_images/0209.png) |

### Labels
Check [flower_images/flower_labels.csv](flower_images/flower_labels.csv)

| Label | 0  | 1  | 2  | 3  | 4  | 5  | 6  | 7  | 8  | 9  | **Total** |
| ----- | -- | -- | -- | -- | -- | -- | -- | -- | -- | -- | --------- |
| Count | 21 | 20 | 19 | 22 | 20 | 25 | 23 | 15 | 26 | 18 | **209**   |


In [1]:
# ~ uncomment to install
# %pip install Pillow panda scikit keras tensorflow

## Image to Feature Vector

The code uses PIL - Python Imaging Library [Pillow](https://pillow.readthedocs.io/en/stable/) to give you insights on how convolutions are applied on images.

[**Popular convolutions in image processing**](<https://en.wikipedia.org/wiki/Kernel_(image_processing)>)

Here we just load an image, and apply filters or matrix convolutions on top of an image.


In [ ]:
from PIL import Image
from PIL import ImageFilter
import numpy as np

# open the image file and convert to matrix

img = Image.open("flower_images/0001.png")
print("Display Original Image")
print("Channel: ", img.mode)
print("Width, Height: ", img.size)
display(img)


# Height, Width, Channel as an array
img_array = np.array(img)
print("Image as an array")

for i in range(0, len(img_array)):
    print(img_array[i])

### Convert Image to Grey Scale

Notice the array difference when working with an RGBA image vs Grey Scale


In [ ]:
# Convert to Grey Scale
img2 = img.convert("L")

print("Gray Scale Image")
# Channel
print("Channel: ", img2.mode)
# Width, Height
print("Width, Height: ", img2.size)
display(img2)

img_array = np.array(img2)  # Height, Width, Channel
print("Image as an array")
for i in range(0, len(img_array)):
    print(img_array[i])

### Applying FIND_EDGES filter (convolution) on the image.


In [ ]:
img3 = img2.filter(ImageFilter.FIND_EDGES)
display(img3)

### Applying a Custom Convolution on an image.


In [ ]:
img3 = img2.filter(ImageFilter.Kernel((3, 3), (-1, -1, -1, -1, 8, -1, -1, -1, -1)))
display(img3)

### A quick helper function to show an image as a vector shape


In [ ]:
def img2vec(f_name, flatten=False):
    """
    Helper function to display an image as a vector
    """
    img = Image.open(f_name)
    img_array = np.array(img)
    if flatten:
        return img_array.flatten()
    else:
        return img_array


fname = "flower_images/0001.png"
a = img2vec(fname)
print("Image Array Dimensions ", a.shape)
print("Array: \n", a)

---


## Linear Image Classification using `scikit`

In this example we are going to apply pre-existing filters or fixed convolutions on the image, and use that the train a linear classifier

Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.

[**SVM Classification**](https://scikit-learn.org/stable/modules/svm.html#svm-classification)

Try a set of combinations with image type (RBGA vs L) and with and without fixed convolutions to train a model, you can see how each of them behave, and if that affects the accuracy of results


In [ ]:
from sklearn import svm
from sklearn.model_selection import cross_val_score

# import svm_cross_validate

# open the flowers...
f = open("flower_images/flower_labels.csv")
names = f.readlines()

data = []
labels = []

### 1. Train Data with fixed convolutions

Take a model with fixed convolutions and see the accuracy of score


In [ ]:
for i in range(1, len(names), 1):
    names[i] = names[i].strip("\n")
    l = names[i].split(",")
    labels.append(int(l[1]))
    img = Image.open("flower_images/" + l[0])
    img = img.convert("L")
    img1 = np.array(img.filter(ImageFilter.FIND_EDGES)).flatten()
    img2 = np.array(img.filter(ImageFilter.DETAIL)).flatten()
    img3 = np.array(img.filter(ImageFilter.CONTOUR)).flatten()
    img4 = np.array(img.filter(ImageFilter.SHARPEN)).flatten()
    img5 = np.array(img.filter(ImageFilter.EDGE_ENHANCE)).flatten()
    img6 = np.array(img.filter(ImageFilter.EDGE_ENHANCE_MORE)).flatten()
    img7 = np.array(img.filter(ImageFilter.SMOOTH_MORE)).flatten()
    img8 = np.array(img.filter(ImageFilter.BLUR)).flatten()
    img9 = np.array(img.filter(ImageFilter.EMBOSS)).flatten()
    data.append(np.concatenate((img1, img2, img3, img4, img5, img6, img7, img8, img9)))

print(
    "Total Labels: ",
    len(labels),
    " | Total Data: ",
    len(data),
    " <-- make sure they match!",
)
print("sample: ", len(data[0]))

clf = svm.LinearSVC(max_iter=100, verbose=10)
scores = cross_val_score(clf, data, labels, cv=5)
print("scores: ", scores)
print("average score: ", sum(scores) / len(scores))
# svmcrossvalidate.getbestC(data, labels)

### 2. Train Data with random convolutions

Take a model with random convolutions and see the accuracy of score


In [ ]:
data = []
labels = []
for i in range(1, len(names), 1):
    names[i] = names[i].strip("\n")
    l = names[i].split(",")
    labels.append(int(l[1]))
    img = Image.open("flower_images/" + l[0])
    img = img.convert("L")
    img_array = np.array(img)
    data.append(img_array.flatten())

print(
    "Total Labels: ",
    len(labels),
    " | Total Data: ",
    len(data),
    " <-- make sure they match!",
)

print("sample: ", len(data[0]))

#### Running LinearSVC

Regularized along with weights, with multi-class classification of one vs the rest

Try different `loss` functions, and play around with `max_iter` to check the change in results


In [ ]:
clf = svm.LinearSVC(max_iter=10000, loss="hinge", verbose=10)
scores = cross_val_score(clf, data, labels, cv=5)
print("scores: ", scores)
print("average score: ", sum(scores) / len(scores))
# svmcrossvalidate.getbestC(data, labels)

#### Running LinearSVC

[LinearSVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html)

> Similar to SVC with parameter kernel=’linear’, but implemented in terms of liblinear rather than libsvm, so it has more flexibility in the choice of penalties and loss functions and should scale better to large numbers of samples.
>
> The main differences between LinearSVC and SVC lie in the loss function used by default, and in the handling of intercept regularization between those two implementations.

Regularized along with weights, with multi-class classification of one vs the rest

Try different `loss` functions, and play around with `max_iter` to check the change in results


In [ ]:
clf = svm.SVC(kernel="poly", degree=2, verbose=True)
scores = cross_val_score(clf, data, labels, cv=5)
print("scores: ", scores)
print("average score: ", sum(scores) / len(scores))

#### Running Linear

[SVC(Support Vector Classification)](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html)
Another `kernel` for SVC

Try different `loss` functions, and play around with `max_iter` to check the change in results


In [ ]:
clf = svm.SVC(kernel="linear", verbose=True)
scores = cross_val_score(clf, data, labels, cv=5)
print("scores: ", scores)
print("average score: ", sum(scores) / len(scores))

Let's try to identify the best potential value of $C$

> This can take up to a minute or two, depending upon your laptop


---


## Convoluted Neural Networks


In [19]:
from PIL import Image
import numpy as np
import keras
import pandas as pd
import tensorflow as tf
from keras.layers import Dense, Dropout, Activation, Flatten
from keras.layers import Conv2D, MaxPool2D, AveragePooling2D
from keras.models import Sequential

num_classes = 10
epochs = 500
batch_size = 4

img_rows, img_cols = 32, 32

df = pd.read_csv("flower_images/flower_labels.csv")  # load images' names and labels
names = df["file"].values
labels = df["label"].values.reshape(-1, 1)

print("names: ", len(names))
print("labels: ", len(labels))

data = []

for name in names:
    img = Image.open("flower_images/" + name)  # shape is 128x128x4
    img = img.resize((img_rows, img_cols))  # resize image into fixed size 32x32x4
    img = np.array(img)[np.newaxis, :, :, :3]  # add new axis and new size is 1x32x32x3
    data.append(img)

data = np.concatenate(data)  # concatenate images, shape is 209x128x128x3

print("data loaded: ", len(data))

# print(data[:1])

names:  209
labels:  209
data loaded:  209


In [20]:
x_train = np.array(data).astype(np.float32)/255
# convert label into one-hot vector
y_train = tf.keras.utils.to_categorical(labels, num_classes)


In [27]:
model = Sequential(
    [
        keras.Input(shape=[32, 32, 3]),
        keras.layers.Conv2D(filters=32, kernel_size=(3, 3), padding="valid", activation="relu"),
        keras.layers.MaxPooling2D(pool_size=(30, 30)),
        keras.layers.Flatten(),
        keras.layers.Dense(10),
        keras.layers.Activation("softmax"),
    ]
)

model.compile(
    loss=keras.losses.CategoricalCrossentropy(),
    optimizer=keras.optimizers.Adam(learning_rate=1e-3, weight_decay=1e-6),
    metrics=[keras.metrics.CategoricalAccuracy()],
)

# Instead of tuning things by hand, use some factors and math to decide when to stop, based on callbacks!
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.05, patience=5, min_lr=1e-5),
]

model.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)               │ (None, 30, 30, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 1, 1, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_9 (Flatten)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 10)             │           330 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_9 (Activation)       │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,226 (4.79 KB)

 Trainable params: 1,226 (4.79 KB)

 Non-trainable params: 0 (0.00 B)

In [28]:
def train():
    model.fit(
        x_train,
        y_train,
        batch_size=batch_size,
        epochs=epochs,
        validation_split=0.15,
        shuffle=True,
        callbacks=callbacks
    )
    
train()

Epoch 1/500
45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - categorical_accuracy: 0.1695 - loss: 2.3056 - val_categorical_accuracy: 0.1875 - val_loss: 2.2660 - learning_rate: 0.0010
Epoch 2/500
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - categorical_accuracy: 0.2599 - loss: 2.2386 - val_categorical_accuracy: 0.2500 - val_loss: 2.2390 - learning_rate: 0.0010
Epoch 3/500
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - categorical_accuracy: 0.2994 - loss: 2.1899 - val_categorical_accuracy: 0.1875 - val_loss: 2.2204 - learning_rate: 0.0010
Epoch 4/500
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - categorical_accuracy: 0.3220 - loss: 2.1434 - val_categorical_accuracy: 0.1875 - val_loss: 2.1714 - learning_rate: 0.0010
Epoch 5/500
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - categorical_accuracy: 0.3390 - loss: 2.0954 - val_categorical_accuracy: 0.1875 - val_loss: 2.1431 - learning_rate: 0.0010
Epoch 6/500
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - categorical_accuracy: 0.3672 - loss: 2.0422 - val_categorical_accu